# 01 — Exploratory Data Analysis (EDA)
## NIDS — Network Intrusion Detection System
**Dataset:** CICIDS2017 Cleaned & Preprocessed (Kaggle)  
**Goal:** Understand the dataset before training — class distribution, feature statistics, correlations, and key patterns.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Nice plot style
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
sns.set_palette('husl')

print('All libraries imported successfully!')


## Step 1 — Load the Dataset


In [ ]:
# Load the cleaned CICIDS2017 dataset
DATA_PATH = '../data/raw/cicids2017_cleaned.csv'

df = pd.read_csv(DATA_PATH)

# Strip whitespace from column names (common CICIDS2017 issue)
df.columns = df.columns.str.strip()

print(f'Dataset loaded successfully!')
print(f'Shape: {df.shape[0]:,} rows x {df.shape[1]} columns')
print(f'Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB')


## Step 2 — First Look at the Data


In [ ]:
# First 5 rows
print('First 5 rows:')
df.head()


In [ ]:
# Column names and data types
print(f'Total columns: {len(df.columns)}')
print('\nColumn names:')
for i, col in enumerate(df.columns):
    print(f'  {i+1:2d}. {col}')


In [ ]:
# Basic info
df.info()


## Step 3 — Identify Label Column


In [ ]:
# Find the label column
label_col = None
for col in df.columns:
    if 'label' in col.lower():
        label_col = col
        break

print(f'Label column found: "{label_col}"')
print(f'\nUnique classes: {df[label_col].nunique()}')
print('\nAll classes:')
for cls in sorted(df[label_col].unique()):
    count = (df[label_col] == cls).sum()
    pct = count / len(df) * 100
    print(f'  {cls:<40} {count:>8,}  ({pct:.2f}%)')


## Step 4 — Class Distribution (Most Important EDA Step)


In [ ]:
class_counts = df[label_col].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Bar chart
colors = ['#2ecc71' if 'BENIGN' in str(c).upper() else '#e74c3c' for c in class_counts.index]
axes[0].bar(range(len(class_counts)), class_counts.values, color=colors, edgecolor='white', linewidth=0.5)
axes[0].set_xticks(range(len(class_counts)))
axes[0].set_xticklabels(class_counts.index, rotation=45, ha='right', fontsize=9)
axes[0].set_title('Class Distribution (Absolute Count)', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Number of Samples')
for i, v in enumerate(class_counts.values):
    axes[0].text(i, v + max(class_counts)*0.01, f'{v:,}', ha='center', fontsize=7, rotation=90)

# Pie chart (log scale for visibility)
axes[1].pie(
    class_counts.values,
    labels=class_counts.index,
    autopct=lambda p: f'{p:.1f}%' if p > 1 else '',
    startangle=90,
    textprops={'fontsize': 8}
)
axes[1].set_title('Class Distribution (Percentage)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.suptitle('CICIDS2017 — Class Distribution', y=1.02, fontsize=16, fontweight='bold')
plt.savefig('../data/processed/eda_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved!')


## Step 5 — Class Imbalance Analysis
This is critical! If BENIGN traffic is 80%+ of data, a model can cheat by just predicting BENIGN always.


In [ ]:
total = len(df)
benign_count = (df[label_col].str.upper() == 'BENIGN').sum()
attack_count = total - benign_count

print('=' * 50)
print(f'Total samples   : {total:>10,}')
print(f'BENIGN traffic  : {benign_count:>10,}  ({benign_count/total*100:.1f}%)')
print(f'Attack traffic  : {attack_count:>10,}  ({attack_count/total*100:.1f}%)')
print('=' * 50)

# Imbalance ratio
ratio = benign_count / max(attack_count, 1)
print(f'\nImbalance ratio: {ratio:.1f}:1  (BENIGN:Attack)')

if ratio > 5:
    print('\n⚠️  HIGH IMBALANCE — SMOTE oversampling required before training!')
elif ratio > 2:
    print('\n⚠️  MODERATE IMBALANCE — class_weight="balanced" recommended')
else:
    print('\n✅ Classes are relatively balanced')


## Step 6 — Missing Values & Data Quality


In [ ]:
print('--- Missing Values ---')
missing = df.isnull().sum()
missing = missing[missing > 0]
if len(missing) == 0:
    print('✅ No missing values found!')
else:
    print(missing)

print('\n--- Infinite Values ---')
numeric_df = df.select_dtypes(include=[np.number])
inf_count = np.isinf(numeric_df).sum().sum()
if inf_count == 0:
    print('✅ No infinite values found!')
else:
    print(f'⚠️  Found {inf_count} infinite values')
    inf_cols = np.isinf(numeric_df).sum()
    print(inf_cols[inf_cols > 0])

print('\n--- Duplicate Rows ---')
dup_count = df.duplicated().sum()
if dup_count == 0:
    print('✅ No duplicate rows found!')
else:
    print(f'⚠️  Found {dup_count:,} duplicate rows ({dup_count/len(df)*100:.2f}%)')


## Step 7 — Statistical Summary of Features


In [ ]:
# Statistical summary of numeric columns
numeric_df = df.select_dtypes(include=[np.number])
print(f'Total numeric features: {len(numeric_df.columns)}')
numeric_df.describe().T.round(3)


## Step 8 — Feature Distributions (Top Important Features)


In [ ]:
# Show distributions of key network features
key_features = []
for candidate in ['Flow Duration', 'Total Fwd Packets', 'Total Backward Packets',
                   'Flow Bytes/s', 'Flow Packets/s', 'Fwd Packet Length Mean',
                   'Bwd Packet Length Mean', 'SYN Flag Count']:
    # Match ignoring case and spaces
    for col in df.columns:
        if col.strip().lower() == candidate.lower():
            key_features.append(col)
            break

# Fallback: just take first 8 numeric columns
if len(key_features) < 4:
    key_features = list(numeric_df.columns[:8])

key_features = key_features[:8]
print(f'Plotting distributions for: {key_features}')

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for i, feat in enumerate(key_features):
    data = df[feat].replace([np.inf, -np.inf], np.nan).dropna()
    # Clip extreme outliers for better visualization
    p99 = data.quantile(0.99)
    data = data[data <= p99]
    axes[i].hist(data, bins=50, color='#3498db', edgecolor='none', alpha=0.8)
    axes[i].set_title(feat, fontsize=9, fontweight='bold')
    axes[i].set_xlabel('Value')
    axes[i].set_ylabel('Count')

plt.suptitle('Feature Distributions (clipped at 99th percentile)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/processed/eda_feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()


## Step 9 — Correlation Heatmap
Highly correlated features (>0.95) can be removed — they carry duplicate information.


In [ ]:
# Use a sample for speed if dataset is large
sample_size = min(50000, len(df))
df_sample = df.sample(sample_size, random_state=42)

numeric_sample = df_sample.select_dtypes(include=[np.number])
numeric_sample = numeric_sample.replace([np.inf, -np.inf], np.nan).fillna(0)

# Take top 20 features by variance for a readable heatmap
top_features = numeric_sample.var().nlargest(20).index
corr_matrix = numeric_sample[top_features].corr()

plt.figure(figsize=(16, 12))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask,
    annot=True, fmt='.2f', annot_kws={'size': 7},
    cmap='RdYlGn', center=0,
    linewidths=0.5, square=True,
    vmin=-1, vmax=1
)
plt.title('Feature Correlation Heatmap (Top 20 Features by Variance)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/processed/eda_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Heatmap saved!')


In [ ]:
# Find pairs of features with correlation > 0.95
threshold = 0.95
corr_full = numeric_sample.corr().abs()

upper = corr_full.where(np.triu(np.ones(corr_full.shape), k=1).astype(bool))
highly_correlated = [
    (col, row, corr_full.loc[row, col])
    for col in upper.columns
    for row in upper.index
    if upper.loc[row, col] > threshold
]

print(f'Feature pairs with correlation > {threshold}:')
if highly_correlated:
    for f1, f2, corr in sorted(highly_correlated, key=lambda x: -x[2]):
        print(f'  {f1:<35} ↔  {f2:<35}  r={corr:.3f}')
    print(f'\nTotal: {len(highly_correlated)} highly correlated pairs')
    print('\n→ These will be handled during preprocessing in notebook 02')
else:
    print('✅ No highly correlated pairs found!')


## Step 10 — Attack vs Benign Traffic Comparison


In [ ]:
# Compare feature means for BENIGN vs all attacks
df_copy = df.copy()
df_copy = df_copy.replace([np.inf, -np.inf], np.nan).dropna()
df_copy['is_attack'] = (~df_copy[label_col].str.upper().str.contains('BENIGN')).astype(int)

# Pick a few key features to compare
compare_features = [c for c in numeric_df.columns[:6] if c in df_copy.columns]

benign_means = df_copy[df_copy['is_attack']==0][compare_features].mean()
attack_means = df_copy[df_copy['is_attack']==1][compare_features].mean()

comparison = pd.DataFrame({'BENIGN': benign_means, 'ATTACK': attack_means})
# Normalize for visualization
comparison_norm = (comparison - comparison.min()) / (comparison.max() - comparison.min() + 1e-9)

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(compare_features))
width = 0.35
ax.bar(x - width/2, comparison_norm['BENIGN'], width, label='BENIGN', color='#2ecc71', alpha=0.8)
ax.bar(x + width/2, comparison_norm['ATTACK'], width, label='ATTACK', color='#e74c3c', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(compare_features, rotation=30, ha='right', fontsize=9)
ax.set_title('BENIGN vs ATTACK — Normalized Feature Means', fontsize=14, fontweight='bold')
ax.set_ylabel('Normalized Mean Value')
ax.legend()
plt.tight_layout()
plt.savefig('../data/processed/eda_benign_vs_attack.png', dpi=150, bbox_inches='tight')
plt.show()


## Step 11 — Box Plots: Feature Spread by Class


In [ ]:
# Box plots for top 4 features grouped by attack type
# Use a sample for speed
df_box = df.replace([np.inf, -np.inf], np.nan).dropna().sample(min(20000, len(df)), random_state=42)

plot_features = [c for c in numeric_df.columns[:4]]

fig, axes = plt.subplots(1, 4, figsize=(20, 6))

for i, feat in enumerate(plot_features):
    # Clip for readability
    p95 = df_box[feat].quantile(0.95)
    data_clipped = df_box[df_box[feat] <= p95]

    classes = data_clipped[label_col].unique()
    data_by_class = [data_clipped[data_clipped[label_col]==cls][feat].values for cls in classes]

    bp = axes[i].boxplot(data_by_class, patch_artist=True, labels=[c[:10] for c in classes])
    colors_bp = ['#2ecc71' if 'BENIGN' in str(cls).upper() else '#e74c3c' for cls in classes]
    for patch, color in zip(bp['boxes'], colors_bp):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)

    axes[i].set_title(feat, fontsize=9, fontweight='bold')
    axes[i].tick_params(axis='x', rotation=90, labelsize=7)

plt.suptitle('Feature Distribution by Class (Box Plots)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/processed/eda_boxplots.png', dpi=150, bbox_inches='tight')
plt.show()


## Step 12 — EDA Summary & Key Findings


In [ ]:
import os
os.makedirs('../data/processed', exist_ok=True)

numeric_df2 = df.select_dtypes(include=[np.number])
class_counts2 = df[label_col].value_counts()
benign_pct = (df[label_col].str.upper() == 'BENIGN').sum() / len(df) * 100

print('=' * 60)
print('  EDA SUMMARY — CICIDS2017 Dataset')
print('=' * 60)
print(f'  Total samples         : {len(df):,}')
print(f'  Total features        : {len(numeric_df2.columns)}')
print(f'  Number of classes     : {df[label_col].nunique()}')
print(f'  BENIGN percentage     : {benign_pct:.1f}%')
print(f'  Missing values        : {df.isnull().sum().sum()}')
print(f'  Infinite values       : {np.isinf(numeric_df2).sum().sum()}')
print(f'  Duplicate rows        : {df.duplicated().sum():,}')
print('=' * 60)
print()
print('KEY FINDINGS:')
print(f'  1. Dataset is highly imbalanced — BENIGN is {benign_pct:.0f}% of data')
print(f'     → SMOTE must be applied before training')
print(f'  2. {len(numeric_df2.columns)} numerical features available for ML models')
print(f'  3. {df[label_col].nunique()} attack classes including BENIGN')
print(f'  4. Highly correlated features should be removed to reduce noise')
print()
print('NEXT STEPS  →  Open notebook 02_training.ipynb')
print('  - Preprocess: drop correlated features, apply SMOTE, scale')
print('  - Train: Logistic Regression, Decision Tree, Random Forest, XGBoost')
print('  - Compare: accuracy, F1-score, precision, recall table')
print('  - Ensemble: Voting Classifier + Stacking')
